# Tutorial 5: Dirac traces with FORM

**What you will learn**
1. traces of gamma matrices in 4 and in $D$ dimensions
2. traces with $\gamma_5$
3. traces with slashed momenta and masses
4. a whole cross section: $e^+e^- \to \mu^+\mu^-$ at tree level
5. the trace of the pion decay $\pi^0 \to \gamma\gamma$

feynsage sends the traces to FORM, the program used for the big multi-loop calculations. FORM must be installed (`./install.sh` does it).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))     # use the feynsage of this repository (not needed after ./install.sh)
from feynsage import *
%display latex

## Step 1: simple traces

`form.trace(indices)` is the trace of $\gamma^{\mu}\gamma^{\nu}\cdots$ in $D$ dimensions. The metric is written `g(mu, nu)`.

In [2]:
form.trace(['mu', 'nu'])

4*g(mu, nu)

In [3]:
form.trace(['mu', 'nu', 'rho', 'sigma'])

4*g(mu, sigma)*g(nu, rho) - 4*g(mu, rho)*g(nu, sigma) + 4*g(mu, nu)*g(rho, sigma)

## Step 2: contracted indices

`form.dirac_trace(factors, vectors, dim)` is more general. A repeated index is summed. In $D$ dimensions $\gamma^\mu\gamma_\mu = D$, so $\mathrm{Tr}[\gamma^\mu\gamma_\mu] = 4D$:

In [4]:
form.dirac_trace(['mu', 'mu'], vectors=[], dim='D')

4*D

## Step 3: $\gamma_5$

`'g5'` is $\gamma_5$. With $\gamma_5$ use `dim=4` (the default). The answer contains the Levi-Civita tensor. FORM's `e_` is $-i$ times the usual $\epsilon^{\mu\nu\rho\sigma}$; feynsage keeps FORM's convention and writes it `eps`.

In [5]:
form.dirac_trace(['g5', 'mu', 'nu', 'rho', 'sigma'], vectors=[])

4*eps(mu, nu, rho, sigma)

In [6]:
form.dirac_trace(['g5', 'mu', 'nu'], vectors=[])     # fewer than four gammas with g5: zero

0

## Step 4: slashed momenta and masses

A factor like `'p + m'` means $\not p + m$. List the momenta in `vectors`. A scalar product comes back as `dot(p, k)`. The trace of a fermion line with a mass:

In [7]:
form.dirac_trace(['p + m', 'mu', 'k + m', 'nu'], vectors=['p', 'k'])

4*m^2*g(mu, nu) + 4*comp(k, nu)*comp(p, mu) + 4*comp(k, mu)*comp(p, nu) - 4*dot(p, k)*g(mu, nu)

This is $4\left[p^\mu k^\nu + p^\nu k^\mu - g^{\mu\nu}(p\cdot k - m^2)\right]$, the trace in every QED textbook.

## Step 5: a full calculation, $e^+e^- \to \mu^+\mu^-$

For massless fermions, summed over spins,
$$\sum|\mathcal M|^2 = \frac{e^4}{s^2}\,\mathrm{Tr}[\not p'\gamma^\mu\not p\gamma^\nu]\;\mathrm{Tr}[\not k\gamma_\mu\not k'\gamma_\nu] .$$
Two traces share the indices $\mu, \nu$. For this we write a few lines of FORM ourselves and run them with `form.run_form`. `g_(1, ...)` and `g_(2, ...)` are the two fermion lines. The `id` lines put in the Mandelstam variables: $p\cdot k = p'\cdot k' = -t/2$ and $p\cdot k' = p'\cdot k = -u/2$.

In [8]:
code = '''
Symbols s, t, u;
Vectors p, pp, k, kp;
Indices mu, nu;
Local M = g_(1, pp, mu, p, nu) * g_(2, k, mu, kp, nu);
trace4, 1;
trace4, 2;
id p.k = -t/2;   id pp.kp = -t/2;
id p.kp = -u/2;  id pp.k = -u/2;
Print;
.end
'''
out = form.run_form(code)
body = out.split("M =")[1].split(";")[0]
traces = SR(" ".join(body.split()))
traces

8*t^2 + 8*u^2

So the two traces give $8(t^2 + u^2)$. Average over the four initial spin states:
$$\frac14\sum|\mathcal M|^2 = \frac{2e^4(t^2 + u^2)}{s^2} .$$
In the centre-of-mass frame $t = -\frac s2(1 - \cos\theta)$ and $u = -\frac s2(1 + \cos\theta)$:

In [9]:
s, t, u, e, th = var('s t u e theta')
msq = (e^4/s^2 * traces / 4).subs(t=-s/2*(1 - cos(th)), u=-s/2*(1 + cos(th)))
msq.simplify_full()

-e^4*sin(theta)^2 + 2*e^4

Since $2 - \sin^2\theta = 1 + \cos^2\theta$, this is the famous $1 + \cos^2\theta$. With $d\sigma/d\Omega = \overline{|\mathcal M|^2}/(64\pi^2 s)$ and $e^2 = 4\pi\alpha$ it gives $d\sigma/d\Omega = \frac{\alpha^2}{4s}(1 + \cos^2\theta)$ (Peskin and Schroeder, eq. 5.12).

In [10]:
alpha = var('alpha')
dsig = (msq / (64*pi^2*s)).subs(e=sqrt(4*pi*alpha)).simplify_full()
print(dsig)
print("total cross section:", integrate(integrate(dsig*sin(th), th, 0, pi), var('phi'), 0, 2*pi))

-1/4*(alpha^2*sin(theta)^2 - 2*alpha^2)/s
total cross section: 4/3*pi*alpha^2/s


The total cross section $4\pi\alpha^2/(3s)$ comes out as well.

## Step 6: the pion trace

The quark loop of $\pi^0 \to \gamma\gamma$ (Section 25 of the note) has the trace below, after the loop momentum is shifted to $q$. Every term with $q$ drops out and only one term, $\pm 4m\,\epsilon(k_1, k_2, \mu, \nu)$, is left (the sign follows the order of $\mu$ and $\nu$ along the loop). This is why the decay rate does not depend on the details of the loop.

In [11]:
form.dirac_trace(['g5', 'q - x*k1 - (1-y)*k2 + m', 'nu', 'q - x*k1 + y*k2 + m', 'mu',
                  'q + (1-x)*k1 + y*k2 + m'], vectors=['q', 'k1', 'k2'])

-4*m*eps(k1, k2, mu, nu)

## Exercises

**1.** Show that in $D$ dimensions $\mathrm{Tr}[\gamma^\mu\gamma^\nu\gamma_\mu\gamma_\nu] = -4D(D - 2)$. (By hand: $\gamma^\mu\gamma^\nu\gamma_\mu = -(D-2)\gamma^\nu$.)
<details><summary>Show a solution</summary>

```python
form.dirac_trace(['mu', 'nu', 'mu', 'nu'], vectors=[], dim='D').factor()
```
</details>

**2.** Compute $\mathrm{Tr}[(\not p + m)\gamma^\mu(\not p' + m)\gamma_\mu]$ in 4 dimensions. You should get $-8\,p\cdot p' + 16m^2$.
<details><summary>Show a solution</summary>

```python
form.dirac_trace(['p + m', 'mu', 'pp + m', 'mu'], vectors=['p', 'pp'])
```
</details>

**3.** Change the FORM code of Step 5 so that the muon has a mass $M$ (put `+ M` on the muon line, `g_(2, k, ...)` becomes `(g_(2,k) + M*gi_(2))`). Which new term appears?
<details><summary>Show a solution</summary>

```python
code_m = '''
Symbols s, t, u, M;
Vectors p, pp, k, kp;
Indices mu, nu;
Local M2 = g_(1, pp, mu, p, nu) * (g_(2,k) + M*gi_(2)) * g_(2, mu) * (g_(2,kp) - M*gi_(2)) * g_(2, nu);
trace4, 1;
trace4, 2;
Print;
.end
'''
print(form.run_form(code_m).split("M2 =")[1].split(";")[0])
```
</details>